In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Installing Libraries

In [1]:
ROLL_NUMBER = '21f2000707'   
SEED = 42

!pip install -q transformers datasets scikit-learn lightgbm sentencepiece

import os, random
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
import lightgbm as lgb
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import torch
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm


random.seed(SEED); np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 10.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 41.5 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pylibcudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
cudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
bigframes 2.12.0 requires google-cloud-bigquery[bqstorage,pandas]>=3.31.0, but you have google-cloud-bigquery 3.25.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.1.0 which is incompatible.
gradio 5.38.1 requires pydantic<2.12,>=2.0, 

2025-10-19 06:39:37.219463: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760855977.430766      37 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760855977.492267      37 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


# Loading Dataset

In [2]:
train = pd.read_csv('/kaggle/input/the-ancient-texts-provenance-challenge/train.csv')
test  = pd.read_csv('/kaggle/input/the-ancient-texts-provenance-challenge/test.csv')
train['text'] = train['text'].fillna('').astype(str)
test['text']  = test['text'].fillna('').astype(str)
print('train', train.shape, 'test', test.shape)

train (119656, 3) test (29914, 2)


# TF-IDF Vectorization

In [3]:
# Encoding labels
le = LabelEncoder()
train['lbl_enc'] = le.fit_transform(train['label'])

#train_sample = train.sample(40000, random_state=SEED)  # smaller sample for quick run; comment out for full dataset
X, y = train['text'].values, train['lbl_enc'].values

# Train/validation split
Xtr, Xv, ytr, yv = train_test_split(X, y, test_size=0.12, stratify=y, random_state=SEED)

# TF-IDF vectorization 
tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), max_features=20000)
Xtr_t = tfidf.fit_transform(Xtr)
Xv_t  = tfidf.transform(Xv)

# LightGBM Training

In [4]:
# LightGBM datasets
dtrain = lgb.Dataset(Xtr_t, label=ytr, free_raw_data=False)
dvalid = lgb.Dataset(Xv_t, label=yv, reference=dtrain, free_raw_data=False)

# LightGBM parameters
params = {
    'objective': 'multiclass',
    'num_class': len(le.classes_),
    'boosting_type': 'gbdt',
    'metric': 'multi_logloss',
    'learning_rate': 0.1,
    'num_leaves': 64,
    'feature_fraction': 0.7,
    'bagging_fraction': 0.8,
    'bagging_freq': 4,
    'seed': SEED,
    'n_jobs': -1
}

In [5]:
# Callbacks for early stopping and logging
callbacks = [
    lgb.early_stopping(stopping_rounds=20),
    lgb.log_evaluation(period=50)
]

# Train the model
gbm = lgb.train(
    params,
    dtrain,
    num_boost_round=200,  # fewer rounds for faster training
    valid_sets=[dvalid],
    callbacks=callbacks
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 17.969273 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3537247
[LightGBM] [Info] Number of data points in the train set: 105297, number of used features: 20000
[LightGBM] [Info] Start training from score -2.575845
[LightGBM] [Info] Start training from score -3.020121
[LightGBM] [Info] Start training from score -3.322574
[LightGBM] [Info] Start training from score -3.002565
[LightGBM] [Info] Start training from score -3.139024
[LightGBM] [Info] Start training from score -1.959054
[LightGBM] [Info] Start training from score -2.797211
[LightGBM] [Info] Start training from score -1.221218
[LightGBM] [Info] Start training from score -1.845396
[LightGBM] [Info] Start training from score -3.898318
[LightGBM] [Info] Start training from score -3.054170
[LightGBM] [Info] Start training from score -

# Predictions

In [6]:
# Validation predictions
pred_val = np.argmax(gbm.predict(Xv_t, num_iteration=gbm.best_iteration), axis=1)
print('LightGBM TF-IDF Macro F1:', f1_score(yv, pred_val, average='macro'))

# Prediction on test set
Xt = tfidf.transform(test['text'].values)
pred_test = np.argmax(gbm.predict(Xt, num_iteration=gbm.best_iteration), axis=1)

LightGBM TF-IDF Macro F1: 0.3947743631535477


# Submission File (Using on LightGBM)

In [7]:
# Saving LightGBM submission
sub = pd.DataFrame({'id': test['id'], 'label': le.inverse_transform(pred_test)})
sub.to_csv('/kaggle/working/submission.csv', index=False)
print('LightGBM submission saved to /kaggle/working/submission.csv')

LightGBM submission saved to /kaggle/working/submission.csv


### This submission gives a score of 0.39 

# Transformer Fine-tuning (Single-split)

In [8]:
# Tokenizer + Dataset
MODEL_NAME = 'xlm-roberta-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# sample subset for quick testing; comment out for full dataset
# train_sample = train.sample(60000, random_state=SEED)
# dataset = Dataset.from_pandas(train_sample)

dataset = Dataset.from_pandas(train)
dataset = dataset.rename_column("label", "labels")  # HF Trainer expects 'labels'

# Tokenization
def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=256)

dataset = dataset.map(tokenize, batched=True, batch_size=512)
dataset = dataset.remove_columns(['text', 'id'])  # remove unnecessary columns
dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# Train/validation split
dataset = dataset.train_test_split(test_size=0.12, seed=SEED)
train_dataset = dataset['train']
val_dataset   = dataset['test']

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Map:   0%|          | 0/119656 [00:00<?, ? examples/s]

In [9]:
# Model
num_labels = train['label'].nunique()
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
# Training Arguments
training_args = TrainingArguments(
    output_dir='./results',
    overwrite_output_dir=True,
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    learning_rate=3e-5,
    logging_steps=100,
    save_steps=5000,
    eval_steps=5000,
    fp16=True,
    seed=SEED,
    report_to=[]   # report_to disables wandb / other logging integrations
)

In [11]:
# Metrics
from sklearn.metrics import f1_score

def compute_metrics(p):
    preds = p.predictions.argmax(axis=-1)
    return {"f1_macro": f1_score(p.label_ids, preds, average='macro')}

In [12]:
# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/tmp/ipykernel_37/785703435.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [13]:
# Train
trainer.train()

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
100,2.272200
200,2.151100
300,2.151400
400,2.034100
500,2.024000
600,1.975000
700,1.933900
800,1.844100
900,1.823100
1000,1.795700


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


TrainOutput(global_step=6582, training_loss=1.4562003194507007, metrics={'train_runtime': 6414.9651, 'train_samples_per_second': 32.829, 'train_steps_per_second': 1.026, 'total_flos': 2.770803854966477e+16, 'train_loss': 1.4562003194507007, 'epoch': 2.0})

# TF-IDF + Transformer

In [14]:
def batch_predict_model_fast(trainer, texts, tokenizer, max_len=256, batch_size=256):
    encodings = tokenizer(texts, truncation=True, padding='max_length', max_length=max_len, return_tensors='pt')
    dataset = TensorDataset(encodings['input_ids'], encodings['attention_mask'])
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    device = trainer.model.device
    trainer.model.eval()
    trainer.model.to(device)
    logits_list = []

    with torch.no_grad():
        for input_ids, attention_mask in tqdm(loader, desc="Predicting Transformer", leave=False):
            input_ids = input_ids.to(device, non_blocking=True)
            attention_mask = attention_mask.to(device, non_blocking=True)
            outputs = trainer.model(input_ids=input_ids, attention_mask=attention_mask)
            logits_list.append(outputs.logits.detach().cpu().numpy())
    return np.vstack(logits_list)


# Predictions

In [15]:
# Transformer predictions
test_texts = test['text'].tolist()
logits_transformer = batch_predict_model_fast(trainer, test_texts, tokenizer, max_len=256, batch_size=256)
proba_transformer = torch.nn.functional.softmax(torch.tensor(logits_transformer), dim=1).numpy()

Predicting Transformer:   0%|          | 0/117 [00:00<?, ?it/s]

In [16]:
# Final labels
final_pred = proba_transformer.argmax(axis=1)
final_labels = le.inverse_transform(final_pred)

# Submission File (Using Tranformers)

In [17]:
submission = pd.DataFrame({'id': test['id'], 'label': final_labels})
submission.to_csv('/kaggle/working/new_submission.csv', index=False)
print("Transformer submission saved to /kaggle/working/new_submission.csv")

Transformer submission saved to /kaggle/working/new_submission.csv
